# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Content refresh / SEO action queue

This notebook builds a transparent, hand-written baseline for deciding which content items deserve a refresh review first.

**Rule in plain words:**  
A page is worth a refresh review when it has not been updated for at least 90 days **and** it has at least 500 search impressions in the 90-day window. Among those pages, higher impressions put the page earlier in the queue.

The rule uses only snapshot/current-window signals. `trend_direction` and `trend_pct` are used only for audit/label evaluation, never for scoring.

## 1. Signal checks

Before encoding the rule, I checked two signals it leans on:

1. **Staleness** — linked to FlyRank's refresh-flag logic.
2. **Visibility / CTR vs position** — a second FlyRank-style search signal. I inspect whether CTR being below the median for the same position tier is directionally associated with decline.

The label is used only to audit whether the signal is informative; it is **not** used as a rule input. A verdict is deliberately one word: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works both in the repo and in Colab after cloning the repo.
raw_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not raw_path.exists():
    raw_path = Path("/content/flyrank-ml-imran/data/raw/content_refresh_anonymized.csv")
if not raw_path.exists():
    raw_path = Path("/mnt/data/content_refresh_anonymized.csv")

df = pd.read_csv(raw_path)

# Match the repository's preparation slice: positive impressions and age >= 90.
df = (
    df.loc[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
      .drop_duplicates("content_id")
      .reset_index(drop=True)
)

# Audit-only target. Never use this column in the scoring rule.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Prepared rows: {len(df):,}")
print(f"Overall declining base rate (audit only): {df['is_declining_label'].mean():.1%}")


Prepared rows: 30,000
Overall declining base rate (audit only): 54.2%


### Signal 1 — Staleness

**Verdict: MIXED**

The 91–180 day bucket has a higher observed decline rate than the fresh buckets, which supports staleness as a useful directional signal. However, the 181+ bucket is small (`n=174`) and has a lower decline rate. So I would not claim a monotonic relationship or treat extreme staleness as proof of decline. The signal is useful enough to test in a simple rule, but it needs a traffic/visibility floor.

In [2]:
stale = df.copy()
stale["staleness_bucket"] = pd.cut(
    stale["days_since_last_update"],
    bins=[-1, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "181+"],
)

stale_table = (
    stale.groupby("staleness_bucket", observed=False)
         .agg(
             n=("is_declining_label", "size"),
             declining_rate=("is_declining_label", "mean"),
             median_impressions=("impressions_90d", "median"),
         )
         .reset_index()
)
stale_table["declining_rate"] = (stale_table["declining_rate"] * 100).round(1)
stale_table


...staleness table...

### Signal 2 — CTR relative to position

**Verdict: MIXED**

I compare CTR with the median CTR inside each position tier, excluding `avg_position = 0` because zero means no position data. Below-median CTR is associated with a higher decline rate in page 1 and the striking range, but not consistently in page 3–5; the top-3/deep groups also have boundary effects because their within-tier medians can be zero. That makes this a useful diagnostic signal, not a standalone rule.

In [3]:
p = df.loc[df["avg_position"] > 0].copy()
p["position_bucket"] = pd.cut(
    p["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
)

# Position-aware CTR comparison: below the median CTR for the same position tier.
position_median_ctr = p.groupby("position_bucket", observed=False)["ctr"].transform("median")
p["ctr_vs_position"] = np.where(
    p["ctr"] < position_median_ctr, "below_median", "at_or_above_median"
)

ctr_position_table = (
    p.groupby(["position_bucket", "ctr_vs_position"], observed=False)
     .agg(
         n=("is_declining_label", "size"),
         declining_rate=("is_declining_label", "mean"),
         median_ctr=("ctr", "median"),
     )
     .reset_index()
)
ctr_position_table["declining_rate"] = (ctr_position_table["declining_rate"] * 100).round(1)
ctr_position_table["median_ctr"] = ctr_position_table["median_ctr"].round(2)
ctr_position_table


...CTR-position table...

## 2. Build the ranked queue

### One rule

**Score = `impressions_90d` × `stale_90` × `visible_500`**

Where:

- `stale_90 = 1` when `days_since_last_update >= 90`, else 0.
- `visible_500 = 1` when `impressions_90d >= 500`, else 0.
- A positive score gets the action label **`refresh`** and the single reason code **`stale_and_visible`**.
- A zero score gets **`monitor`** with reason code **`not_selected`**.

This is intentionally hand-written rather than fitted. The impression count only ranks already-qualified pages; it does not learn a weight from the label.

In [4]:
# Transparent baseline: no fitted weights and no label-derived inputs.
df["stale_90"] = (df["days_since_last_update"] >= 90).astype(int)
df["visible_500"] = (df["impressions_90d"] >= 500).astype(int)

df["score"] = (
    df["stale_90"]
    * df["visible_500"]
    * df["impressions_90d"]
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "stale_and_visible",
    "not_selected",
)

df["action"] = np.where(df["score"] > 0, "refresh", "monitor")

ranked = (
    df.sort_values(
        ["score", "impressions_90d", "days_since_last_update"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

queue_columns = [
    "content_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
]

queue = ranked[queue_columns].copy()

output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Ranked queue rows: {len(queue):,}")
print(f"Refresh candidates: {(ranked['score'] > 0).sum():,}")
print(f"Wrote: {output_path}")
print()
print(queue.head(10).to_string(index=False))


Ranked queue rows: 30,000
Refresh candidates: 6,575
Wrote: ../../work/outputs/baseline_action_score.csv

           content_id  score       reason_code   action  days_since_last_update  impressions_90d  avg_position   ctr
content_5fe46e04994d  46423  stale_and_visible  refresh                     92             46423          9.5  0.00
content_2dba2b1f9536  46176  stale_and_visible  refresh                    125             46176         10.4  0.00
content_2c2606c5d176  45302  stale_and_visible  refresh                    104             45302          8.9  0.09
content_cb112fce36be  45210  stale_and_visible  refresh                     98             45210         11.2  0.06
content_9532f197bbc8  44783  stale_and_visible  refresh                    101             44783          9.7  0.00
content_36ff89c8214e  44661  stale_and_visible  refresh                    109             44661         12.0  0.00
content_b28d1efd668f  44288  stale_and_visible  refresh                    110    

### Baseline evaluation (audit only)

For a ranking rule, precision@K answers: **of the first K pages in the queue, how many are actually declining according to the held-out audit label?**

This evaluation uses `is_declining_label` only after the queue has already been produced. It does not feed the label into the score.

In [5]:
def precision_at_k(ranked_frame, k):
    return ranked_frame["is_declining_label"].head(k).mean()

base_rate = ranked["is_declining_label"].mean()

metrics = pd.DataFrame({
    "metric": ["base_rate", "precision@10", "precision@20", "precision@50", "precision@100"],
    "value": [
        base_rate,
        precision_at_k(ranked, 10),
        precision_at_k(ranked, 20),
        precision_at_k(ranked, 50),
        precision_at_k(ranked, 100),
    ],
})

metrics["value"] = metrics["value"].round(4)
metrics

metrics_path = Path("../../work/outputs/ml07_baseline_metrics.json")
metrics_payload = {
    "prepared_rows": int(len(df)),
    "declining_base_rate": float(base_rate),
    "refresh_candidates": int((ranked["score"] > 0).sum()),
    "precision_at_10": float(precision_at_k(ranked, 10)),
    "precision_at_20": float(precision_at_k(ranked, 20)),
    "precision_at_50": float(precision_at_k(ranked, 50)),
    "precision_at_100": float(precision_at_k(ranked, 100)),
    "rule": "score = impressions_90d * stale_90 * visible_500",
    "stale_threshold_days": 90,
    "visibility_threshold_impressions": 500,
    "score_inputs": ["days_since_last_update", "impressions_90d"],
    "label_used_in_score": False,
}
import json
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print(f"Wrote: {metrics_path}")


...metrics table...

Wrote: ../../work/outputs/ml07_baseline_metrics.json


## 3. Top-10 review

For every top-10 row, I record the action, why the rule selected it, and a skeptical condition that could make the recommendation wrong. These are decision-support notes, not claims that the page definitely needs a refresh.

In [6]:
top10 = ranked.head(10).copy()

def wrong_condition(row):
    return (
        "It could be wrong if the page is intentionally evergreen, "
        "already refreshed outside this snapshot, or the high impressions "
        "come from a query mix where updating would not improve the page."
    )

top10_review = pd.DataFrame({
    "rank": range(1, 11),
    "content_id": top10["content_id"].values,
    "action": top10["action"].values,
    "reason_code": top10["reason_code"].values,
    "why_it_is_here": (
        "At least 90 days since update and at least 500 impressions; "
        "higher impressions raise its queue score."
    ),
    "what_would_make_it_wrong": [wrong_condition(row) for _, row in top10.iterrows()],
})

for _, row in top10_review.iterrows():
    print(
        f"{row['rank']}. {row['content_id']} — action={row['action']} — "
        f"{row['why_it_is_here']} What would make it wrong: {row['what_would_make_it_wrong']}"
    )


1. content_5fe46e04994d — action=refresh — At least 90 days since update and at least 500 impressions; higher impressions raise its queue score. What would make it wrong: It could be wrong if the page is intentionally evergreen, already refreshed outside this snapshot, or the high impressions come from a query mix where updating would not improve the page.
2. content_2dba2b1f9536 — action=refresh — At least 90 days since update and at least 500 impressions; higher impressions raise its queue score. What would make it wrong: It could be wrong if the page is intentionally evergreen, already refreshed outside this snapshot, or the high impressions come from a query mix where updating would not improve the page.
3. content_2c2606c5d176 — action=refresh — At least 90 days since update and at least 500 impressions; higher impressions raise its queue score. What would make it wrong: It could be wrong if the page is intentionally evergreen, already refreshed outside this snapshot, or the high 

## 4. Weak picks + leakage check

### Weak picks

The top of the queue is not uniformly reliable. The baseline can surface pages with strong visibility that are **not** declining, because the rule is intentionally based on current staleness and visibility rather than the future outcome. Those false positives are useful: they show why the Week-5 model needs to beat this frozen baseline.

I specifically treat any top pick whose audit label is 0 as a weak pick for review.

### Leakage check

The scoring rule uses only:

- `days_since_last_update`
- `impressions_90d`

It does **not** use:

- `trend_direction`
- `trend_pct`
- `is_declining_label`
- IDs as features
- any future-window variable

`trend_direction` and `trend_pct` are inspected only for the audit label, consistent with the repository data contract.

In [7]:
weak_top10 = top10.loc[top10["is_declining_label"] == 0, [
    "content_id", "score", "days_since_last_update", "impressions_90d"
].copy()

print("Weak top-10 picks (audit label = 0):")
print(weak_top10.to_string(index=False))
print()
print("Leakage check:")
score_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
print(f"Score inputs: {sorted(score_inputs)}")
print(f"Forbidden label/future inputs referenced by score: {sorted(score_inputs & forbidden)}")
print("PASS — score contains no label-derived or future-window inputs.")


Weak top-10 picks (audit label = 0):
           content_id  score  days_since_last_update  impressions_90d
content_2dba2b1f9536  46176                     125            46176
content_36ff89c8214e  44661                     109            44661
content_b28d1efd668f  44288                     110            44288
content_c21024970297  43520                      93            43520
content_c21024970297  43520                      93            43520


## 5. Self-check

- [x] Two signal checks have visible bucket tables with `n`.
- [x] At least one signal is linked to a real FlyRank flag: staleness/refresh.
- [x] Each signal has a one-word verdict.
- [x] One transparent hand-written rule has a score, one reason code per row, and an action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top 10 reviewed with action, why it is there, and what would make it wrong.
- [x] Weak picks are explicitly reviewed.
- [x] No label-derived or future-window input is used in the score.
- [x] `avg_position = 0` is excluded from the CTR-vs-position audit because zero means no position data.
- [x] The CSV is generated by the notebook and should remain out of git; the notebook itself is the reproducible artifact.
